# KoELECTRA ONNX → Float32 TFLite (No FlexErf)

**목적**: int8_no_erf 실패 대비 float32 백업 모델 생성  
**방식**: onnx2tf → SavedModel → TFLiteConverter(TFLITE_BUILTINS) → float32  
**결과**: `model_float32_no_erf.tflite` (약 430 MB, 양자화 없음)

**사전 준비**: Drive `내 드라이브/vp_model/` 에 `model.onnx`, `model.onnx.data` 업로드  
**실행 순서**: Cell 1 실행 → 런타임 재시작 → Cell 2 → Cell 3 → Cell 4

In [ ]:
# Cell 1: 패키지 설치
!pip install -q onnx2tf sng4onnx onnxsim onnx onnxruntime transformers
print('설치 완료 - 런타임 재시작 후 Cell 2부터 실행')

In [ ]:
# Cell 2: Drive 마운트 + 파일 복사
from google.colab import drive
import shutil, os
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/vp_model'
TMP_DIR   = '/tmp/onnx_src'
os.makedirs(TMP_DIR, exist_ok=True)
shutil.copy(f'{DRIVE_DIR}/model.onnx',      f'{TMP_DIR}/model.onnx')
shutil.copy(f'{DRIVE_DIR}/model.onnx.data', f'{TMP_DIR}/model.onnx.data')
print(f'복사 완료: {os.path.getsize(TMP_DIR+"/model.onnx")/1024**2:.1f} MB')

In [ ]:
# Cell 3: ONNX -> SavedModel -> Float32 TFLite (TFLITE_BUILTINS 전용)
# 양자화 없음 → 가중치 float32 유지
# TFLITE_BUILTINS 만 사용 → FlexErf 대신 TFLite 네이티브 ERF op 사용
import shutil, os, glob
import tensorflow as tf
import onnx2tf

TMP_DIR    = '/tmp/onnx_src'
OUT_DIR    = '/tmp/onnx2tf_out'
SM_DIR     = '/tmp/saved_model_f32'
OUT_TFLITE = '/tmp/model_float32_no_erf.tflite'

shutil.rmtree(OUT_DIR, ignore_errors=True)
shutil.rmtree(SM_DIR,  ignore_errors=True)
os.makedirs(OUT_DIR)

print('onnx2tf 변환 중 (수 분 소요)...')
tf_model = onnx2tf.convert(
    input_onnx_file_path=f'{TMP_DIR}/model.onnx',
    output_folder_path=OUT_DIR,
    overwrite_input_shape=[
        'input_ids:1,128',
        'attention_mask:1,128',
        'token_type_ids:1,128',
    ],
    non_verbose=True,
)
print(f'반환값 타입: {type(tf_model)}')

if tf_model is not None:
    print('SavedModel 저장 중...')
    tf.saved_model.save(tf_model, SM_DIR)

    print('Float32 TFLite 변환 중...')
    converter = tf.lite.TFLiteConverter.from_saved_model(SM_DIR)
    # 양자화 없음 (float32 유지)
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
    try:
        tflite = converter.convert()
        print('성공: TFLITE_BUILTINS (FlexDelegate 불필요)')
    except Exception as e:
        print(f'TFLITE_BUILTINS 실패: {e}')
        print('SELECT_TF_OPS 포함으로 재시도...')
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS,
            tf.lite.OpsSet.SELECT_TF_OPS,
        ]
        tflite = converter.convert()
        print('성공 (FlexErf 포함됨 - Android에서 실패할 수 있음)')

    with open(OUT_TFLITE, 'wb') as f:
        f.write(tflite)
    print(f'완료: {os.path.getsize(OUT_TFLITE)/1024**2:.1f} MB')
else:
    print('[!] tf_model=None - OUT_DIR 파일:')
    for f in sorted(glob.glob(f'{OUT_DIR}/*.tflite')):
        print(f'  {f}  ({os.path.getsize(f)/1024**2:.1f} MB)')

In [ ]:
# Cell 4: Drive 저장
import shutil, os
DRIVE_DIR  = '/content/drive/MyDrive/vp_model'
OUT_TFLITE = '/tmp/model_float32_no_erf.tflite'
if os.path.exists(OUT_TFLITE):
    dst = f'{DRIVE_DIR}/model_float32_no_erf.tflite'
    shutil.copy(OUT_TFLITE, dst)
    print(f'저장: {dst}  ({os.path.getsize(dst)/1024**2:.1f} MB)')
    print('다운로드 후 client/assets/model_float32_no_erf.tflite 로 복사')
else:
    print('파일 없음 - Cell 3 확인 필요')